# Activity 1: RAGAS Evaluation with Cost Analysis

Compare the Fireworks-hosted RAG pipeline (`gpt-oss-20b` + `qwen3-embedding-8b`) against an
OpenAI-hosted equivalent (`gpt-4.1-mini` + `text-embedding-3-small`) on the same cat health PDF.

We score both with RAGAS (faithfulness, context precision, context recall) using a fixed
OpenAI judge so the evaluation itself doesn't favor either provider, and we compute a
cost-per-query breakdown from real published pricing plus actual token usage.

Note: RAGAS's `AnswerRelevancy` metric hangs indefinitely in the installed version (0.4.3) due
to an upstream bug in its embeddings-based self-consistency check. It's excluded here; the
other three metrics still cover retrieval quality, faithfulness, and end-to-end accuracy.

LangSmith tracing follows `LANGSMITH_TRACING` in `.env`. When enabled, every
retrieval/generation call lands in the LangSmith project `10-llm-servers-ragas-eval`, where the
tracing and cost dashboards can be cross-checked against the manual token/cost math below. It's
currently off by default because traces from repeated eval runs add up on LangSmith's billing —
flip it on for one demo run if you want traces to show in the Loom.

In [1]:
import os
import sys
import types
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
load_dotenv(Path("../09_Agent_Servers/.env"), override=False)

os.environ.setdefault("LANGSMITH_PROJECT", "10-llm-servers-ragas-eval")

vertexai_stub = types.ModuleType("langchain_community.chat_models.vertexai")
vertexai_stub.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules["langchain_community.chat_models.vertexai"] = vertexai_stub

## Eval set

Two question/reference pairs hand-written from the AAHA/AAFP Feline Life Stage Guidelines PDF,
covering life stages and vaccination. Kept small deliberately: each
RAGAS judge call genuinely takes 30-120s, and running them concurrently causes timeouts (see the
note on the RAGAS evaluation cell below), so this eval runs fully serially.

In [2]:
EVAL_QUESTIONS = [
    {
        "question": "What are the four life stages defined in the 2021 AAHA/AAFP Feline Life Stage Guidelines, and their age ranges?",
        "reference": "The guidelines define four life stages: kitten (birth to 1 year), young adult (1 to 6 years), mature adult (7 to 10 years), and senior (over 10 years), plus a separate end-of-life stage.",
    },
    {
        "question": "What are the core vaccines recommended for cats?",
        "reference": "The core vaccines are rabies virus, feline herpesvirus type 1 (FHV-1), feline calicivirus (FCV), and feline panleukopenia virus (FPV). Feline leukemia virus (FeLV) vaccination is also considered core for kittens and young cats due to age-related susceptibility.",
    },
]

## Shared document loading and chunking (identical for both pipelines)

In [3]:
import tiktoken
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def tiktoken_len(text):
    return len(tiktoken.encoding_for_model("gpt-4o").encode(text))


loader = DirectoryLoader("data", glob="**/*.pdf", loader_cls=PyMuPDFLoader)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=750, chunk_overlap=0, length_function=tiktoken_len)
chunks = splitter.split_documents(documents)
len(chunks)

/var/folders/km/bn2rsdrd7qd_dl9y2xxk4jqh0000gq/T/ipykernel_88550/3806035135.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader


42

## Pricing

Pulled from Fireworks' and OpenAI's published pricing pages (per 1M tokens).

In [4]:
PRICING = {
    "fireworks": {
        "chat_input_per_1m": 0.07,
        "chat_output_per_1m": 0.30,
        "embedding_per_1m": 0.10,
    },
    "openai": {
        "chat_input_per_1m": 0.40,
        "chat_output_per_1m": 1.60,
        "embedding_per_1m": 0.02,
    },
}

## RAG pipeline

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

RAG_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "human",
            "\n#CONTEXT:\n{context}\n\nQUERY:\n{query}\n\n"
            "Use the provided context to answer the query. Only use the provided context. "
            'If the answer is not in the context, say "I don\'t know".',
        )
    ]
)


def format_context(docs):
    return "\n\n".join(d.page_content for d in docs)


class RagPipeline:
    def __init__(self, name, embedding_model, chat_model, base_url, api_key, embedding_dimensions=None, k=4):
        self.name = name
        embedding_kwargs = dict(
            model=embedding_model,
            openai_api_key=api_key,
            openai_api_base=base_url,
            check_embedding_ctx_length=False,
        )
        if embedding_dimensions:
            embedding_kwargs["dimensions"] = embedding_dimensions
        self.embeddings = OpenAIEmbeddings(**embedding_kwargs)
        self.vectorstore = QdrantVectorStore.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            location=":memory:",
            collection_name=f"rag_{name}",
        )
        self.retriever = self.vectorstore.as_retriever(search_kwargs={"k": k})
        self.llm = ChatOpenAI(model=chat_model, openai_api_key=api_key, openai_api_base=base_url, temperature=0)

    def run(self, question):
        retrieved = self.retriever.invoke(question)
        context_text = format_context(retrieved)
        raw_response = (RAG_PROMPT | self.llm).invoke({"query": question, "context": context_text})
        usage = raw_response.usage_metadata or {}
        return {
            "answer": raw_response.content,
            "contexts": [d.page_content for d in retrieved],
            "input_tokens": usage.get("input_tokens", 0),
            "output_tokens": usage.get("output_tokens", 0),
        }

In [6]:
fireworks_pipeline = RagPipeline(
    name="fireworks",
    embedding_model=os.environ["FIREWORKS_EMBEDDING_MODEL"],
    chat_model=os.environ["FIREWORKS_CHAT_MODEL"],
    base_url="https://api.fireworks.ai/inference/v1",
    api_key=os.environ["FIREWORKS_API_KEY"],
    embedding_dimensions=4096,
)

openai_pipeline = RagPipeline(
    name="openai",
    embedding_model="text-embedding-3-small",
    chat_model="gpt-4.1-mini",
    base_url="https://api.openai.com/v1",
    api_key=os.environ["OPENAI_API_KEY"],
)

## Run both pipelines over the eval set

In [7]:
def run_pipeline_over_eval(pipeline, questions):
    rows = []
    for item in questions:
        result = pipeline.run(item["question"])
        rows.append(
            {
                "question": item["question"],
                "reference": item["reference"],
                "answer": result["answer"],
                "contexts": result["contexts"],
                "input_tokens": result["input_tokens"],
                "output_tokens": result["output_tokens"],
                "query_embedding_tokens": tiktoken_len(item["question"]),
            }
        )
    return rows


fireworks_rows = run_pipeline_over_eval(fireworks_pipeline, EVAL_QUESTIONS)
openai_rows = run_pipeline_over_eval(openai_pipeline, EVAL_QUESTIONS)

## RAGAS evaluation

Both pipelines are judged by the same OpenAI model so the comparison reflects the pipelines,
not the judge. Uses RAGAS's newer `ragas.metrics.collections` API with a direct `AsyncOpenAI`
client via `llm_factory` — the older `LangchainLLMWrapper` path hangs on every structured-output
metric in this ragas version. `AnswerRelevancy` is excluded because it hangs indefinitely even on
the newer API; the remaining three metrics still cover retrieval quality, faithfulness, and
end-to-end accuracy.

Scoring runs **fully serially, one judge call at a time**. Running these calls concurrently
(even with just 4 in flight) consistently caused `TimeoutError`s that don't occur when each call
runs alone — this is a real bottleneck in this environment, not a fluke. Serial execution is
slower (each call takes 30-120s) but reliable: budget roughly 15-20 minutes for the full run.

In [8]:
import asyncio
import time

from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecisionWithReference, ContextRecall, Faithfulness

judge_client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"], timeout=90)
judge_llm = llm_factory("gpt-4o-mini", client=judge_client)

faithfulness_metric = Faithfulness(llm=judge_llm)
context_precision_metric = ContextPrecisionWithReference(llm=judge_llm)
context_recall_metric = ContextRecall(llm=judge_llm)


async def score_one(name, coro):
    start = time.time()
    try:
        result = await asyncio.wait_for(coro, timeout=180)
        print(f"  {name}: {result.value:.3f} ({time.time() - start:.0f}s)")
        return result.value
    except Exception as e:
        print(f"  {name}: FAILED {type(e).__name__} ({time.time() - start:.0f}s)")
        return None


async def score_rows(rows, label):
    scored = []
    for i, row in enumerate(rows):
        print(f"[{label}] row {i + 1}/{len(rows)}: {row['question'][:60]}")
        faithfulness = await score_one(
            "faithfulness",
            faithfulness_metric.ascore(
                user_input=row["question"], response=row["answer"], retrieved_contexts=row["contexts"]
            ),
        )
        context_precision = await score_one(
            "context_precision",
            context_precision_metric.ascore(
                user_input=row["question"], reference=row["reference"], retrieved_contexts=row["contexts"]
            ),
        )
        context_recall = await score_one(
            "context_recall",
            context_recall_metric.ascore(
                user_input=row["question"], retrieved_contexts=row["contexts"], reference=row["reference"]
            ),
        )
        scored.append(
            {
                "faithfulness": faithfulness,
                "context_precision": context_precision,
                "context_recall": context_recall,
            }
        )
    return scored


fireworks_ragas_scores = await score_rows(fireworks_rows, "fireworks")
openai_ragas_scores = await score_rows(openai_rows, "openai")

[fireworks] row 1/2: What are the four life stages defined in the 2021 AAHA/AAFP 


  faithfulness: 1.000 (68s)


  context_precision: 0.750 (127s)


  context_recall: 1.000 (32s)
[fireworks] row 2/2: What are the core vaccines recommended for cats?


  faithfulness: 1.000 (67s)


...

  context_precision: 0.583 (126s)


  context_recall: 1.000 (32s)
[openai] row 2/2: What are the core vaccines recommended for cats?


  faithfulness: 1.000 (67s)


  context_precision: 1.000 (126s)


  context_recall: 1.000 (32s)


In [9]:
import pandas as pd

ragas_comparison = pd.DataFrame(
    {
        "fireworks (gpt-oss-20b)": pd.DataFrame(fireworks_ragas_scores).mean(numeric_only=True),
        "openai (gpt-4.1-mini)": pd.DataFrame(openai_ragas_scores).mean(numeric_only=True),
    }
)
ragas_comparison

,fireworks (gpt-oss-20b),openai (gpt-4.1-mini)
faithfulness,1.000,1.000000
context_precision,0.875,0.791667
context_recall,1.000,1.000000


## Cost analysis

Indexing cost is the one-time embedding cost for all PDF chunks. Per-query cost is the
marginal cost of one retrieval + generation call — this is the number that matters "at scale"
since indexing amortizes away. Generation token counts come from each provider's real
`usage_metadata`; embedding token counts are estimated with `tiktoken` since neither provider
returns embedding usage through the LangChain wrapper.

In [10]:
def indexing_cost(provider):
    total_tokens = sum(tiktoken_len(c.page_content) for c in chunks)
    return total_tokens / 1_000_000 * PRICING[provider]["embedding_per_1m"]


def per_query_costs(provider, rows):
    price = PRICING[provider]
    costs = []
    for r in rows:
        embed_cost = r["query_embedding_tokens"] / 1_000_000 * price["embedding_per_1m"]
        gen_cost = (
            r["input_tokens"] / 1_000_000 * price["chat_input_per_1m"]
            + r["output_tokens"] / 1_000_000 * price["chat_output_per_1m"]
        )
        costs.append(embed_cost + gen_cost)
    return costs


fireworks_costs = per_query_costs("fireworks", fireworks_rows)
openai_costs = per_query_costs("openai", openai_rows)

cost_summary = pd.DataFrame(
    {
        "fireworks": {
            "one-time indexing cost ($)": indexing_cost("fireworks"),
            "avg cost per query ($)": sum(fireworks_costs) / len(fireworks_costs),
            "cost per 1,000 queries ($)": sum(fireworks_costs) / len(fireworks_costs) * 1000,
        },
        "openai": {
            "one-time indexing cost ($)": indexing_cost("openai"),
            "avg cost per query ($)": sum(openai_costs) / len(openai_costs),
            "cost per 1,000 queries ($)": sum(openai_costs) / len(openai_costs) * 1000,
        },
    }
)
cost_summary

,fireworks,openai
one-time indexing cost ($),0.002421,0.000484
avg cost per query ($),0.000188,0.000940
"cost per 1,000 queries ($)",0.188035,0.939770


## Analysis

_(fill in after reviewing the tables above — talking points for the Loom)_

- Retrieval quality: compare `llm_context_precision_with_reference` and `context_recall` between providers — did one pipeline surface the right chunks more often?
- Faithfulness: did either model hallucinate beyond what the retrieved context supported?
- Cost at scale: multiply `cost per 1,000 queries` out to your expected traffic and compare against the quality delta — is the more expensive provider worth it?
- Cross-check a few rows in the LangSmith project `10-llm-servers-ragas-eval` to confirm the manual token/cost math lines up with what LangSmith recorded.